# บทที่ 5: เรกูลาไรเซชัน (Regularization)

ใน Notebook นี้ เราจะเรียนรู้เทคนิคต่างๆ เพื่อป้องกันการเรียนรู้เกินพอดี (Overfitting) ได้แก่ เรกูลาไรเซชันแบบ L1/L2 ดรอปเอาต์ (Dropout) การหยุดฝึกฝนก่อนกำหนด (Early Stopping) แบตช์นอร์มัลไลเซชัน (Batch Normalization) และการเพิ่มปริมาณข้อมูล (Data Augmentation)

**ศัพท์ที่สำคัญในบทนี้:**

- เวกเตอร์ (vector) — อาร์เรย์หนึ่งมิติ
- เมทริกซ์ (matrix) — อาร์เรย์สองมิติ
- ค่าน้ำหนัก (weight) — พารามิเตอร์ที่ปรับได้
- ค่าไบแอส (bias) — ค่าเลื่อน
- อินพุต (input) — ข้อมูลนำเข้า
- เอาต์พุต (output) — ผลลัพธ์
- ค่าสูญเสีย (loss) — วัดความคลาดเคลื่อน
- เกรเดียนต์ (gradient) — ทิศทางการปรับ
- ฟังก์ชันกระตุ้น (activation function) — ฟังก์ชันไม่เป็นเชิงเส้น
- โครงข่ายประสาทเทียม (neural network) — โมเดลแมชชีนเลิร์นนิง

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
try:
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'],
                   capture_output=True)
except FileNotFoundError:
    pass  # เครื่องที่ไม่มี apt-get (macOS/Windows) ใช้ฟอนต์ไทยที่ติดตั้งไว้ในเครื่องแทน

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 2. สร้างข้อมูลตัวอย่าง

In [ ]:
# สร้างข้อมูล make_moons
X, y = make_moons(n_samples=500, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', alpha=0.7)
plt.xlabel('คุณลักษณะที่ 1')
plt.ylabel('คุณลักษณะที่ 2')
plt.title('ข้อมูลจำลอง Make Moons')
plt.show()

## 3. L1 และ L2 เรกูลาไรเซชัน (Regularization)

- **L1 (Lasso)**: $\lambda \sum |w_i|$ - ทำให้ค่าน้ำหนักบางตัวเป็น 0
- **L2 (Ridge)**: $\lambda \sum w_i^2$ - ลดขนาดค่าน้ำหนักทุกตัว

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

class MLP_Regularized:
    """เพอร์เซ็ปตรอนหลายชั้นพร้อมเรกูลาไรเซชันแบบ L1/L2"""

    def __init__(self, layer_sizes, learning_rate=0.5, l1_lambda=0.0, l2_lambda=0.0):
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.l1_lambda = l1_lambda
        self.l2_lambda = l2_lambda

        self.weights = []
        self.biases = []

        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * 0.5
            b = np.zeros((layer_sizes[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, x):
        self.activations = [x.reshape(-1, 1)]
        self.z_values = []

        current = self.activations[0]
        for i in range(len(self.layer_sizes) - 1):
            z = np.dot(self.weights[i], current) + self.biases[i]
            self.z_values.append(z)
            current = sigmoid(z)
            self.activations.append(current)

        return current

    def backward(self, y):
        y = y.reshape(-1, 1)
        # เกรเดียนต์ของชั้นเอาต์พุต: dL/dz = (a - y) * f'(z) ตาม chain rule ของค่าสูญเสียกำลังสอง
        delta = (self.activations[-1] - y) * sigmoid_derivative(self.z_values[-1])

        self.dW = []
        self.db = []

        for i in range(len(self.layer_sizes) - 2, -1, -1):
            # เพิ่มพจน์เรกูลาไรเซชันเข้าไปในเกรเดียนต์: L2 ต้องมีตัวคูณ 2 ตามสมการ 2*lambda*w ของบทนี้
            l1_grad = self.l1_lambda * np.sign(self.weights[i])
            l2_grad = 2 * self.l2_lambda * self.weights[i]

            dW = np.dot(delta, self.activations[i].T) + l1_grad + l2_grad
            db = delta

            self.dW.insert(0, dW)
            self.db.insert(0, db)

            if i > 0:
                delta = np.dot(self.weights[i].T, delta) * sigmoid_derivative(self.z_values[i-1])

    def update_weights(self):
        for i in range(len(self.layer_sizes) - 1):
            self.weights[i] -= self.lr * self.dW[i]
            self.biases[i] -= self.lr * self.db[i]

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                self.forward(xi)
                self.backward(yi)
                self.update_weights()

    def predict(self, x):
        return self.forward(x)[0, 0]

    def compute_loss(self, X, y):
        total = 0
        for xi, yi in zip(X, y):
            pred = self.predict(xi)
            total += (yi - pred)**2
        return total / len(X)

## 4. เปรียบเทียบไม่ใช้เรกูลาไรเซชัน กับ L2 เรกูลาไรเซชัน

In [ ]:
# ฝึกโดยไม่ใช้เรกูลาไรเซชัน
mlp_no_reg = MLP_Regularized([2, 64, 64, 1], learning_rate=0.5, l1_lambda=0, l2_lambda=0)
mlp_no_reg.train(X_train, y_train, epochs=2000)

# ฝึกด้วย L2 เรกูลาไรเซชัน
mlp_l2 = MLP_Regularized([2, 64, 64, 1], learning_rate=0.5, l1_lambda=0, l2_lambda=0.01)
mlp_l2.train(X_train, y_train, epochs=2000)

print("=== ค่าสูญเสียชุดฝึก ===")
print(f"ไม่ใช้เรกูลาไรเซชัน: {mlp_no_reg.compute_loss(X_train, y_train):.4f}")
print(f"L2 เรกูลาไรเซชัน: {mlp_l2.compute_loss(X_train, y_train):.4f}")

print("\n=== ค่าสูญเสียชุดทดสอบ ===")
print(f"ไม่ใช้เรกูลาไรเซชัน: {mlp_no_reg.compute_loss(X_test, y_test):.4f}")
print(f"L2 เรกูลาไรเซชัน: {mlp_l2.compute_loss(X_test, y_test):.4f}")

## 5. การสร้างดรอปเอาต์ (Dropout)

In [ ]:
class MLP_Dropout:
    """เพอร์เซ็ปตรอนหลายชั้นพร้อมดรอปเอาต์"""

    def __init__(self, layer_sizes, learning_rate=0.5, dropout_rate=0.5):
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.dropout_rate = dropout_rate

        self.weights = []
        self.biases = []

        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * 0.5
            b = np.zeros((layer_sizes[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, x, training=True):
        self.activations = [x.reshape(-1, 1)]
        self.z_values = []
        self.dropout_masks = []

        current = self.activations[0]
        for i in range(len(self.layer_sizes) - 1):
            z = np.dot(self.weights[i], current) + self.biases[i]
            self.z_values.append(z)
            current = sigmoid(z)

            # ใช้ดรอปเอาต์ (ยกเว้นชั้นเอาต์พุต)
            if training and i < len(self.layer_sizes) - 2:
                mask = (np.random.rand(*current.shape) > self.dropout_rate) / (1 - self.dropout_rate)
                current = current * mask
                self.dropout_masks.append(mask)

            self.activations.append(current)

        return current

    def backward(self, y):
        y = y.reshape(-1, 1)
        # เกรเดียนต์ของชั้นเอาต์พุต: dL/dz = (a - y) * f'(z) ตาม chain rule ของค่าสูญเสียกำลังสอง
        delta = (self.activations[-1] - y) * sigmoid_derivative(self.z_values[-1])

        self.dW = []
        self.db = []

        mask_idx = 0
        for i in range(len(self.layer_sizes) - 2, -1, -1):
            dW = np.dot(delta, self.activations[i].T)
            db = delta

            self.dW.insert(0, dW)
            self.db.insert(0, db)

            if i > 0:
                delta = np.dot(self.weights[i].T, delta) * sigmoid_derivative(self.z_values[i-1])
                if mask_idx < len(self.dropout_masks):
                    delta = delta * self.dropout_masks[-(mask_idx + 1)]
                    mask_idx += 1

    def update_weights(self):
        for i in range(len(self.layer_sizes) - 1):
            self.weights[i] -= self.lr * self.dW[i]
            self.biases[i] -= self.lr * self.db[i]

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                self.forward(xi, training=True)
                self.backward(yi)
                self.update_weights()

    def predict(self, x):
        return self.forward(x, training=False)[0, 0]

## 6. การหยุดฝึกฝนก่อนกำหนด (Early Stopping)

In [ ]:
def train_with_early_stopping(model, X_train, y_train, X_val, y_val,
                               max_epochs=5000, patience=50):
    """
    ฝึกโมเดลพร้อมการหยุดฝึกฝนก่อนกำหนด

    พารามิเตอร์:
    - patience: จำนวนรอบที่รอก่อนหยุดถ้าค่าสูญเสียชุดตรวจสอบไม่ลดลง
    """
    best_val_loss = float('inf')
    best_weights = None
    counter = 0

    train_losses = []
    val_losses = []

    for epoch in range(max_epochs):
        # ฝึกหนึ่งรอบ
        for xi, yi in zip(X_train, y_train):
            model.forward(xi)
            model.backward(yi)
            model.update_weights()

        # คำนวณค่าสูญเสีย
        train_loss = model.compute_loss(X_train, y_train)
        val_loss = model.compute_loss(X_val, y_val)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # ตรวจว่าดีขึ้นหรือไม่
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = [w.copy() for w in model.weights]
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"หยุดฝึกฝนก่อนกำหนดที่รอบ {epoch}")
                break

    # กู้คืนค่าน้ำหนักที่ดีที่สุด
    if best_weights:
        model.weights = best_weights

    return train_losses, val_losses

# เพิ่มเมธอด compute_loss
def compute_loss(self, X, y):
    total = 0
    for xi, yi in zip(X, y):
        pred = self.predict(xi)
        total += (yi - pred)**2
    return total / len(X)

MLP_Dropout.compute_loss = compute_loss

## 7. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: คำนวณค่าปรับ L2 (L2 Penalty)

In [ ]:
# ให้ค่าน้ำหนัก = [0.5, -0.3, 0.8, 0.2] และ λ = 0.01
# จงคำนวณค่าปรับ L2

weights = np.array([0.5, -0.3, 0.8, 0.2])
l2_lambda = 0.01

l2_penalty = l2_lambda * np.sum(weights**2)
print(f"ค่าน้ำหนัก: {weights}")
print(f"ค่าปรับ L2 (λ=0.01): {l2_penalty:.6f}")

# ตรวจคำตอบตามแบบฝึกหัดท้ายบท (ข้อ 1): ค่าน้ำหนัก = [2.0, -1.5, 3.0], λ = 0.01
weights_book = np.array([2.0, -1.5, 3.0])
l2_penalty_book = l2_lambda * np.sum(weights_book**2)
print(f"\nตรวจคำตอบแบบฝึกหัดท้ายบท (w={weights_book}): ค่าปรับ L2 = {l2_penalty_book:.4f}")

### แบบฝึกหัดที่ 2: คำนวณค่าปรับ L1 (L1 Penalty)

In [ ]:
# ให้ค่าน้ำหนักเดียวกัน คำนวณค่าปรับ L1

l1_lambda = 0.01
l1_penalty = l1_lambda * np.sum(np.abs(weights))
print(f"ค่าน้ำหนัก: {weights}")
print(f"ค่าปรับ L1 (λ=0.01): {l1_penalty:.6f}")

# ตรวจคำตอบตามแบบฝึกหัดท้ายบท (ข้อ 2): ค่าน้ำหนัก = [2.0, -1.5, 3.0], λ = 0.01
weights_book = np.array([2.0, -1.5, 3.0])
l1_penalty_book = l1_lambda * np.sum(np.abs(weights_book))
print(f"\nตรวจคำตอบแบบฝึกหัดท้ายบท (w={weights_book}): ค่าปรับ L1 = {l1_penalty_book:.4f}")

### แบบฝึกหัดที่ 3: มาสก์ดรอปเอาต์ (Dropout Mask)

In [ ]:
# ให้ค่ากระตุ้น = [0.8, 0.3, 0.6, 0.9], dropout_rate = 0.5 และมาสก์ตามที่โจทย์ท้ายบทกำหนด m = [1, 0, 1, 0]
# จงคำนวณค่ากระตุ้นหลังใช้ดรอปเอาต์แบบผกผัน

activations = np.array([[0.8], [0.3], [0.6], [0.9]])
dropout_rate = 0.5

# ใช้มาสก์ตายตัวตามที่โจทย์กำหนด (ไม่สุ่ม) เพื่อให้เทียบคำตอบกับบทได้ตรงกัน
mask = np.array([[1], [0], [1], [0]]) / (1 - dropout_rate)
dropped_activations = activations * mask

print(f"ค่ากระตุ้นเดิม:\n{activations.flatten()}")
print(f"\nมาสก์ดรอปเอาต์ (m=[1,0,1,0] ตามโจทย์):\n{mask.flatten()}")
print(f"\nค่ากระตุ้นหลังดรอปเอาต์:\n{dropped_activations.flatten()}")

### แบบฝึกหัดที่ 4: เปรียบเทียบเรกูลาไรเซชัน

In [ ]:
# ทดลองฝึกโมเดล 3 แบบและเปรียบเทียบ

# 1. ไม่ใช้เรกูลาไรเซชัน
mlp1 = MLP_Regularized([2, 32, 1], l1_lambda=0, l2_lambda=0)
mlp1.train(X_train, y_train, epochs=1000)

# 2. L2 เรกูลาไรเซชัน
mlp2 = MLP_Regularized([2, 32, 1], l1_lambda=0, l2_lambda=0.01)
mlp2.train(X_train, y_train, epochs=1000)

# 3. L1 เรกูลาไรเซชัน
mlp3 = MLP_Regularized([2, 32, 1], l1_lambda=0.01, l2_lambda=0)
mlp3.train(X_train, y_train, epochs=1000)

print("=== ค่าสูญเสียชุดฝึก ===")
print(f"ไม่ใช้เรกูลาไรเซชัน: {mlp1.compute_loss(X_train, y_train):.4f}")
print(f"L2: {mlp2.compute_loss(X_train, y_train):.4f}")
print(f"L1: {mlp3.compute_loss(X_train, y_train):.4f}")

print("\n=== ค่าสูญเสียชุดทดสอบ ===")
print(f"ไม่ใช้เรกูลาไรเซชัน: {mlp1.compute_loss(X_test, y_test):.4f}")
print(f"L2: {mlp2.compute_loss(X_test, y_test):.4f}")
print(f"L1: {mlp3.compute_loss(X_test, y_test):.4f}")

## 8. แบตช์นอร์มัลไลเซชัน (Batch Normalization)

In [ ]:
def batch_norm(x, gamma, beta, eps=1e-5):
    """
    แบตช์นอร์มัลไลเซชันตามสูตรในบทนี้:
    mu_B = ค่าเฉลี่ยของแบตช์, sigma_B^2 = ความแปรปรวนของแบตช์
    x_hat = (x - mu_B) / sqrt(sigma_B^2 + eps)
    y = gamma * x_hat + beta
    """
    mu_B = np.mean(x)
    sigma_B2 = np.mean((x - mu_B) ** 2)
    x_hat = (x - mu_B) / np.sqrt(sigma_B2 + eps)
    y = gamma * x_hat + beta
    return y, mu_B, sigma_B2, x_hat

# ทวนตัวอย่างการคำนวณในบท: มินิแบตช์ = [2, 4, 6, 8], eps = 1e-5, gamma = 1.5, beta = 0.5
batch = np.array([2.0, 4.0, 6.0, 8.0])
gamma, beta = 1.5, 0.5
y_bn, mu_B, sigma_B2, x_hat = batch_norm(batch, gamma, beta)

print(f"มินิแบตช์: {batch}")
print(f"mu_B = {mu_B:.4f}")
print(f"sigma_B^2 = {sigma_B2:.4f}")
print(f"x_hat (ค่าเฉลี่ย ~0, ส่วนเบี่ยงเบนมาตรฐาน ~1): {x_hat}")
print(f"y = gamma*x_hat + beta (gamma={gamma}, beta={beta}): {y_bn}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(range(len(batch)), batch, color='steelblue')
axes[0].set_title('ก่อนแบตช์นอร์มัลไลเซชัน')
axes[0].set_xlabel('ตัวอย่างในแบตช์')
axes[0].set_ylabel('ค่า x')

axes[1].bar(range(len(y_bn)), y_bn, color='darkorange')
axes[1].set_title('หลังแบตช์นอร์มัลไลเซชัน (คูณ gamma บวก beta)')
axes[1].set_xlabel('ตัวอย่างในแบตช์')
axes[1].set_ylabel('ค่า y')

plt.tight_layout()
plt.show()

## 9. การเพิ่มปริมาณข้อมูล (Data Augmentation)

In [ ]:
from scipy import ndimage

# สร้างภาพสังเคราะห์รูปตัว L เพื่อให้เห็นผลของการหมุนและพลิกภาพชัดเจน (ภาพไม่สมมาตร)
base_level = 0.15
img = np.full((60, 60), base_level)
img[10:40, 10:20] = 0.9   # แกนตั้งของตัว L
img[30:40, 10:35] = 0.9   # แกนนอนของตัว L

# การแปลงตามที่บทระบุ: หมุน พลิกแนวนอน ปรับความสว่าง เลื่อนตำแหน่ง และขยาย
rotated = ndimage.rotate(img, 15, reshape=False, cval=base_level)
flipped = np.fliplr(img)
brightened = np.clip(img * 1.2, 0, 1)
shifted = ndimage.shift(img, shift=(0, 6), cval=base_level)

zoomed_raw = ndimage.zoom(img, 1.1)
# ตัดกลับให้ขนาดเท่าภาพเดิมเพื่อแสดงในกริดเดียวกัน
zh, zw = zoomed_raw.shape
oy, ox = (zh - 60) // 2, (zw - 60) // 2
zoomed = zoomed_raw[oy:oy+60, ox:ox+60]

images = [img, rotated, flipped, brightened, shifted, zoomed]
titles = ['ภาพต้นฉบับ', 'หมุน 15 องศา', 'พลิกแนวนอน',
          'ปรับความสว่าง +20%', 'เลื่อนตำแหน่ง', 'ขยาย 1.1 เท่า']

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
for ax, im, title in zip(axes, images, titles):
    ax.imshow(im, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

fig.suptitle('การเพิ่มปริมาณข้อมูล: ทุกภาพยังคงป้ายกำกับเดิม (ภาพเดียวกัน แปลงต่างกัน)')
plt.tight_layout()
plt.show()

## บทสรุป

Notebook นี้ครอบคลุม:
1. **เรกูลาไรเซชันแบบ L1**: ทำให้ค่าน้ำหนักบางตัวเป็น 0 (เบาบาง)
2. **เรกูลาไรเซชันแบบ L2**: ลดขนาดค่าน้ำหนักทุกตัว
3. **ดรอปเอาต์ (Dropout)**: สุ่มปิดเซลล์ประสาทระหว่างการฝึก
4. **การหยุดฝึกฝนก่อนกำหนด (Early Stopping)**: หยุดฝึกเมื่อค่าสูญเสียชุดตรวจสอบไม่ลดลง
5. **แบตช์นอร์มัลไลเซชัน (Batch Normalization)**: ปรับค่าเฉลี่ยและความแปรปรวนของมินิแบตช์แล้วปรับขนาด/เลื่อนตำแหน่งด้วยพารามิเตอร์ที่เรียนรู้ได้
6. **การเพิ่มปริมาณข้อมูล (Data Augmentation)**: สร้างตัวอย่างฝึกใหม่จากการแปลงภาพที่รักษาความหมายของป้ายกำกับเดิม